<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_6_integrador_eurosat_ssl.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_6_integrador_eurosat_ssl.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F04_representation%2Fnotebooks%2Fsesion_04_6_integrador_eurosat_ssl.ipynb)

# SI7011 · Sesión 04 · Ejercicio integrador: ¿se puede aprender a mirar la Tierra sin etiquetas?
**Juan David Martínez Vargas · Universidad EAFIT**

En el integrador de la sesión 03 usaste un backbone de **ImageNet** para clasificar imágenes satelitales de EuroSAT con todas las etiquetas. Aquí cambiamos la pregunta: si solo tuvieras el **1 %** o el **10 %** de las etiquetas, ¿te sirve más un encoder preentrenado con fotos comunes **con** etiquetas (ImageNet) o uno preentrenado con las propias imágenes satelitales **sin** etiquetas (SimCLR, notebook 4)?

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | contrato y partición (listos, los mismos de la sesión 03) | `data_card.json` |
| Preentrenamiento | las vistas para imágenes satelitales (**TODO 1**) y la pérdida InfoNCE (**TODO 2**); SimCLR sin etiquetas, registrado en MLflow | `encoder.pt` |
| Evaluación | $k$-NN y *linear probe* con 1 % y 10 % de las etiquetas (**TODO 3**) | tabla en MLflow |
| Despliegue | calcular $\mathbf h$ para imágenes crudas desde MLflow (**TODO 4**) | `embeddings.npz` |

**Duración:** 60–75 min. **Entorno:** Kaggle con GPU (agrega el dataset `apollo2506/eurosat-dataset`) o Colab con GPU. El preentrenamiento tarda unos 10 minutos en una T4.

In [ ]:
%pip install -q mlflow

In [ ]:
import copy, hashlib, json, math, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp = device.type == 'cuda'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· MLflow', mlflow.__version__, '· device', device)

## 1. Datos: contrato, ficha y partición (listo)
El mismo contrato y la misma partición estratificada 70/15/15 con `random_state=0` del integrador de la sesión 03, así que los números son comparables con los tuyos de entonces. La normalización es la de ImageNet para todos los encoders.

In [ ]:
def load_eurosat():
    hits = [p.parent for r in [Path('/kaggle/input'), Path('.')] if r.exists() for p in r.glob('**/AnnualCrop')
            if p.is_dir()]
    if hits:
        print('Leyendo', hits[0]); return datasets.ImageFolder(hits[0]), str(hits[0])
    print('EuroSAT no encontrado en disco: descargando con torchvision.datasets.EuroSAT')
    return datasets.EuroSAT('data', download=True), 'torchvision.datasets.EuroSAT'

ds, src = load_eurosat()
CLASSES = ds.classes; C = len(CLASSES)
CONTRACT = {'mode': 'RGB', 'size': [64, 64]}

def validate_image(img, contract=CONTRACT):
    if img.mode != contract['mode']:
        raise ValueError(f"Se esperaba una imagen {contract['mode']}, llegó {img.mode}.")
    if list(img.size) != contract['size']:
        raise ValueError(f"Se esperaba {contract['size']}, llegó {list(img.size)}.")
    return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)        # uint8, 3 × 64 × 64

t0 = time.time()
X_all = torch.stack([validate_image(ds[i][0]) for i in range(len(ds))])
y_all = torch.tensor([label for _, label in ds.samples])
names = sorted(Path(p).name for p, _ in ds.samples)
data_card = {'fuente': src, 'sha256_nombres': hashlib.sha256('\n'.join(names).encode()).hexdigest()[:16],
             'imagenes': len(ds), 'clases': CLASSES, 'contrato': CONTRACT}
print(f'{len(ds)} imágenes validadas en {time.time() - t0:.0f} s')

idx_all = np.arange(len(y_all))
idx_train, idx_rest = train_test_split(idx_all, test_size=0.30, stratify=y_all, random_state=0)
idx_val, idx_test = train_test_split(idx_rest, test_size=0.50, stratify=y_all[idx_rest], random_state=0)
X_train, y_train = X_all[idx_train].to(device), y_all[idx_train].to(device)
X_val, y_val = X_all[idx_val].to(device), y_all[idx_val].to(device)
X_test, y_test = X_all[idx_test].to(device), y_all[idx_test].to(device)
print('entrenamiento', len(y_train), '· validación', len(y_val), '· prueba', len(y_test))

MEAN = ResNet18_Weights.IMAGENET1K_V1.transforms().mean
STD = ResNet18_Weights.IMAGENET1K_V1.transforms().std
_m, _s = torch.tensor(MEAN, device=device).view(1, 3, 1, 1), torch.tensor(STD, device=device).view(1, 3, 1, 1)
norm = lambda x: (x - _m) / _s                                  # x ∈ [0, 1] → normalizado con μ, s de ImageNet
prep = lambda xb: norm(xb.float() / 255)                        # uint8 → normalizado

**Las etiquetas de entrenamiento no se usan en el preentrenamiento.** Solo aparecen en la evaluación (TODO 3), y solo una fracción de ellas.

## 2. TODO 1 · Dos vistas de una imagen satelital · 10 min
Escribe `vista_satelite(xb)`: recibe un lote `uint8` de $N \times 3 \times 64 \times 64$ en la GPU y devuelve una vista aleatoria **normalizada** (usa `norm`) del mismo tamaño, con parámetros distintos para cada imagen. Dos llamadas sobre el mismo lote deben dar vistas distintas.

Elige las transformaciones con el criterio de la diapositiva *Augmentations define "the same"*: lo que la vista cambia es lo que $\mathbf h$ aprenderá a **ignorar**. Piensa en EuroSAT:
- una foto tomada desde arriba no tiene "arriba": ¿qué rotaciones y espejos son válidos?
- ¿el color distingue un bosque de un cultivo, o un río de una autopista? ¿Cuánto *color jitter* te puedes permitir?

Puedes reutilizar `recorte_espejo`, `color_jitter` y `gris` del notebook 4; están copiadas abajo.

In [ ]:
# del notebook 4 (listas para usar)
def recorte_espejo(x, escala=(0.2, 1.0), razon=(3 / 4, 4 / 3)):
    B = len(x)
    area = torch.empty(B, device=x.device).uniform_(*escala)
    r = torch.exp(torch.empty(B, device=x.device).uniform_(math.log(razon[0]), math.log(razon[1])))
    w, h = (area * r).sqrt().clamp(max=1), (area / r).sqrt().clamp(max=1)
    cx = (torch.rand(B, device=x.device) * 2 - 1) * (1 - w); cy = (torch.rand(B, device=x.device) * 2 - 1) * (1 - h)
    espejo = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 0, 2], theta[:, 1, 1], theta[:, 1, 2] = w * espejo, cx, h, cy
    return F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), mode='bilinear',
                         padding_mode='reflection', align_corners=False)

def gris(x):
    return (0.299 * x[:, :1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:]).expand_as(x)

def color_jitter(x, b=0.4, c=0.4, s=0.4, hue=0.1, p=0.8):
    B = len(x); u = lambda a: torch.empty(B, 1, 1, 1, device=x.device).uniform_(1 - a, 1 + a)
    y = x * u(b)
    m = gris(y).mean((1, 2, 3), keepdim=True); y = (y - m) * u(c) + m
    gy = gris(y); y = (y - gy) * u(s) + gy
    yiq = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]], device=x.device)
    ang = torch.empty(B, device=x.device).uniform_(-hue, hue) * 2 * math.pi
    rot = torch.zeros(B, 3, 3, device=x.device); rot[:, 0, 0] = 1
    rot[:, 1, 1], rot[:, 1, 2], rot[:, 2, 1], rot[:, 2, 2] = ang.cos(), -ang.sin(), ang.sin(), ang.cos()
    y = torch.einsum('bij,bjhw->bihw', torch.linalg.inv(yiq) @ rot @ yiq, y).clamp(0, 1)
    return torch.where(torch.rand(B, 1, 1, 1, device=x.device) < p, y, x)

In [ ]:
def vista_satelite(xb):
    # TODO 1: x̃ = t(x), t ∼ 𝒯, una vista por imagen; uint8 → normalizada con norm(·)
    raise NotImplementedError('Completa TODO 1')

In [ ]:
xb = X_train[:16]
v1, v2 = vista_satelite(xb), vista_satelite(xb)
assert v1.shape == (16, 3, 64, 64) and v1.dtype == torch.float32, 'Esperaba un lote float de 16 × 3 × 64 × 64.'
assert not torch.allclose(v1, v2), 'Dos llamadas deben dar vistas distintas.'
assert abs(v1.mean().item()) < 1.5, '¿Normalizaste con norm(·)?'
ver = lambda t: (t * _s + _m).clamp(0, 1).permute(0, 2, 3, 1).cpu()
fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for c in range(8):
    for r, im in enumerate([xb.permute(0, 2, 3, 1).cpu() / 255, ver(v1), ver(v2)]):
        axes[r, c].imshow(im[c]); axes[r, c].axis('off')
axes[0, 0].set_title(CLASSES[y_train[0]], fontsize=7)
plt.tight_layout(); plt.show()

## 3. TODO 2 · La pérdida InfoNCE · 5 min
Escribe `info_nce(u, tau)` a partir de la fórmula de la diapositiva:
$$s_{ij} = \frac{\mathbf u_i^\top \mathbf u_j}{\tau\,\lVert\mathbf u_i\rVert\,\lVert\mathbf u_j\rVert}, \qquad L_i = -\log \frac{\exp(s_{i,j(i)})}{\sum_{k \ne i} \exp(s_{ik})}$$
`u` tiene $2N$ filas: las $N$ primeras son la vista 1 de cada imagen y las $N$ siguientes, la vista 2, así que $j(i) = i + N$ para $i < N$ y $j(i) = i - N$ para $i \ge N$. Devuelve el promedio de $L_i$ y el acierto del positivo (la fracción de filas cuyo $s_{ik}$ más alto es el de $j(i)$).

In [ ]:
def info_nce(u, tau):
    # TODO 2: devuelve (pérdida promedio, acierto del positivo)
    raise NotImplementedError('Completa TODO 2')

In [ ]:
torch.manual_seed(0)
loss, acierto = info_nce(torch.randn(8, 16), 0.5)
assert abs(float(loss) - 2.2823) < 1e-3, f'Con u aleatorio esperaba 2.2823, obtuve {float(loss):.4f}: revisa s_ij, la diagonal o j(i).'
e = torch.eye(4, 16)
loss, acierto = info_nce(torch.cat([e, e]), 0.5)                 # las dos vistas idénticas y ortogonales a las demás
esperado = math.log(1 + 6 * math.exp(-2))                         # −2 + log(e² + 6 e⁰)
assert abs(float(loss) - esperado) < 1e-4 and float(acierto) == 1.0, 'Revisa el caso de vistas idénticas.'
print('info_nce correcta')

## 4. Preentrenamiento SimCLR sin etiquetas (listo)
- $g$: la ResNet-18, con la primera convolución de 3×3 sin *stride* para no reducir tan pronto las imágenes de 64×64; $\mathbf h \in \mathbb R^{512}$.
- $p$: Linear–BN–ReLU–Linear, de 512 a 128, solo para entrenar.
- $N = 256$, $\tau = 0.2$, SGD con *warmup* y coseno, 30 épocas sobre las imágenes de entrenamiento **sin sus etiquetas**.

`preentrenar` registra en MLflow los parámetros, la pérdida y el acierto del positivo por época, y guarda `encoder.pt` con $g$, la normalización y el contrato: lo que necesitas para el TODO 4.

In [ ]:
def resnet18_sat():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)   # 64×64 → max pool → 32×32
    m.fc = nn.Identity()                                              # g(x) = h ∈ ℝ^512
    return m

def cabeza():
    return nn.Sequential(nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Linear(512, 128))

mlflow.set_tracking_uri('sqlite:///mlflow.db')
EXPERIMENT = 'S04-eurosat-ssl'
mlflow.set_experiment(EXPERIMENT)

def preentrenar(nombre, vista, params, epocas=30, N=256, tau=0.2, construir=resnet18_sat):
    torch.manual_seed(0)
    g, p = construir().to(device), cabeza().to(device)
    opt = torch.optim.SGD([*g.parameters(), *p.parameters()], lr=0.06 * N / 256, momentum=0.9, weight_decay=5e-4, nesterov=True)
    pasos = len(X_train) // N; total, warm = epocas * pasos, pasos
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: (t + 1) / warm if t < warm else
                                              0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler(enabled=amp)
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.log_params({**params, 'epocas': epocas, 'N': N, 'tau': tau, 'imagenes_sin_etiqueta': len(X_train)})
        mlflow.log_dict(data_card, 'data_card.json')
        for ep in range(epocas):
            t0, perd, acie = time.time(), [], []
            g.train(); p.train()
            for b in torch.randperm(len(X_train), device=device)[:pasos * N].split(N):
                xb = X_train[b]                                           # sin y_train
                with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                    loss, acierto = info_nce(p(g(torch.cat([vista(xb), vista(xb)]))).float(), tau)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                perd.append(loss.item()); acie.append(acierto.item())
            mlflow.log_metrics({'infonce': np.mean(perd), 'acierto_positivo': np.mean(acie), 'epoca_s': time.time() - t0}, step=ep)
            print(f'época {ep + 1:2d} · InfoNCE {np.mean(perd):.3f} · acierto del positivo {np.mean(acie):.3f} · {time.time() - t0:.0f} s')
        torch.save({'g': g.state_dict(), 'model': construir.__name__, 'mean': MEAN, 'std': STD, 'contract': CONTRACT}, 'encoder.pt')
        mlflow.log_artifact('encoder.pt')
    return g, run.info.run_id

**Predice:** ¿qué valor tendrá la pérdida en la primera época? ¿A qué número debería acercarse el acierto del positivo si la red adivinara al azar?

In [ ]:
params = {'vistas': '...',   # describe tu TODO 1 en pocas palabras
          'encoder': 'resnet18_sat', 'cabeza': '512-512-128'}
g_ssl, run_ssl = preentrenar('1-simclr-eurosat', vista_satelite, params)

## 5. TODO 3 · Evaluar con pocas etiquetas · 15 min
Comparamos tres encoders **congelados**:

| Encoder | Etiquetas que usó para preentrenar |
|---|---|
| aleatorio | ninguna |
| ImageNet | 1.2 millones, de fotos comunes |
| SimCLR EuroSAT (tuyo) | ninguna, pero vio imágenes satelitales |

`features(g, X)` ya calcula $\mathbf h$ para un conjunto de imágenes, y `knn` y `probe` son las del notebook 5. Escribe `evaluar(H_train, H_val, frac, seed)`: elige al azar, **por clase**, la fracción `frac` de las imágenes de entrenamiento etiquetadas (usa `seed`), y devuelve `{'knn': ..., 'probe': ...}` con la accuracy en **validación** de las dos lecturas, entrenadas solo con esas imágenes.

In [ ]:
@torch.no_grad()
def features(g, X, chunk=500):
    g.eval()
    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
        return torch.cat([g(prep(X[i:i + chunk])).float() for i in range(0, len(X), chunk)])

@torch.no_grad()
def knn(Htr, ytr, Hev, yev, k=20, t=0.07):
    Htr, Hev = F.normalize(Htr, dim=1), F.normalize(Hev, dim=1); k = min(k, len(Htr))
    sim, j = (Hev @ Htr.T).topk(k, dim=1)
    votos = torch.zeros(len(j), C, device=Hev.device).scatter_add_(1, ytr[j], (sim / t).exp())
    return (votos.argmax(1) == yev).float().mean().item()

def probe(Htr, ytr, Hev, yev, steps=300, seed=0):
    mu, sd = Htr.mean(0), Htr.std(0) + 1e-6
    torch.manual_seed(seed)
    head = nn.Linear(Htr.shape[1], C).to(Htr.device)
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-4)
    for _ in range(steps):
        loss = F.cross_entropy(head((Htr - mu) / sd), ytr); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head((Hev - mu) / sd).argmax(1) == yev).float().mean().item()

g_imnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1); g_imnet.fc = nn.Identity()
torch.manual_seed(0)
ENCODERS = {'aleatorio': resnet18_sat().to(device), 'ImageNet': g_imnet.to(device), 'SimCLR EuroSAT': g_ssl}
H = {n: (features(g, X_train), features(g, X_val)) for n, g in ENCODERS.items()}

In [ ]:
def evaluar(H_train, H_val, frac, seed=0):
    # TODO 3: submuestra estratificada de una fracción frac de las etiquetas de entrenamiento; k-NN y probe en validación
    raise NotImplementedError('Completa TODO 3')

In [ ]:
r = evaluar(*H['ImageNet'], 0.01)
assert set(r) == {'knn', 'probe'} and all(0 <= v <= 1 for v in r.values()), "Esperaba {'knn': acc, 'probe': acc}."
assert evaluar(*H['ImageNet'], 0.01, seed=0) == evaluar(*H['ImageNet'], 0.01, seed=0), 'Con la misma semilla debe dar lo mismo.'
assert r['probe'] > 0.3, 'Con ImageNet y 1 % de las etiquetas esperaba bastante más que el azar (0.10).'

FRACS, SEMILLAS = [0.01, 0.10], [0, 1, 2]
filas = []
for n, (Htr, Hva) in H.items():
    for frac in FRACS:
        rs = [evaluar(Htr, Hva, frac, s) for s in SEMILLAS]
        filas.append({'encoder': n, 'etiquetas': f'{frac:.0%}', 'imagenes_etiquetadas': round(frac * len(y_train)),
                      'knn': np.mean([x['knn'] for x in rs]), 'probe': np.mean([x['probe'] for x in rs])})
tabla = pd.DataFrame(filas)
with mlflow.start_run(run_id=run_ssl):
    mlflow.log_table(tabla, 'evaluacion_pocas_etiquetas.json')
    mlflow.log_metrics({f"val_{f['etiquetas'].rstrip('%')}pct_{m}": f[m] for f in filas if f['encoder'] == 'SimCLR EuroSAT'
                        for m in ['knn', 'probe']})
tabla.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, m in zip(axes, ['knn', 'probe']):
    piv = tabla.pivot(index='encoder', columns='etiquetas', values=m).loc[list(H)]
    piv.plot.bar(ax=ax, rot=0); ax.set_title(m); ax.axhline(1 / C, color='k', ls=':', lw=1)
axes[0].set_ylabel('accuracy de validación'); plt.tight_layout(); plt.show()

## 6. Cambia una sola cosa · 10 min
Elige **un** cambio y córrelo como `'2-...'` con `preentrenar`. Antes de correrlo, escribe en `params` qué esperas que pase y luego repite la evaluación solo para el nuevo encoder. Algunas opciones:

- quita el *color jitter* de tus vistas, o quita las rotaciones;
- $\tau = 0.5$;
- empieza desde los pesos de ImageNet en lugar de pesos aleatorios (`construir` debe devolver la ResNet-18 de ImageNet con `fc = nn.Identity()`): **preentrenamiento continuo** en el dominio;
- la mitad de las épocas.

_Mi cambio y mi predicción:_

In [ ]:
# Copia la celda de preentrenar con un cambio, y luego evalúa: evaluar(features(g2, X_train), features(g2, X_val), 0.01)

## 7. Prueba, una sola vez (listo)
Con la mejor combinación según **validación** (encoder y lectura, con 10 % de las etiquetas), medimos en prueba.

In [ ]:
mejor = tabla[tabla.etiquetas == '10%'].melt(id_vars=['encoder'], value_vars=['knn', 'probe']).sort_values('value').iloc[-1]
print(f"elegido por validación: {mejor.encoder} + {mejor.variable} ({mejor.value:.3f})")
g_best = ENCODERS[mejor.encoder]
Htr, Hte = features(g_best, X_train), features(g_best, X_test)
g_ = torch.Generator().manual_seed(0)
i10 = torch.cat([(y_train == c).nonzero().squeeze(1)[torch.randperm(int((y_train == c).sum()), generator=g_)[:round(0.10 * int((y_train == c).sum()))].to(device)]
                 for c in range(C)])
lectura = knn if mejor.variable == 'knn' else probe
test_acc = lectura(Htr[i10], y_train[i10], Hte, y_test)
with mlflow.start_run(run_id=run_ssl):
    mlflow.set_tag('eleccion_final', f'{mejor.encoder} + {mejor.variable}'); mlflow.log_metric('test_acc_10pct', test_acc)
print(f'prueba: {test_acc:.3f}')

## 8. TODO 4 · Despliegue: embeddings de imágenes crudas · 10 min
Un encoder sin etiquetas se despliega como un **servicio de embeddings**: recibe imágenes y devuelve $\mathbf h$, que otros usan para buscar imágenes parecidas o entrenar su propio clasificador con pocas etiquetas.

Escribe `embed_raw(ckpt, images)`. Recibe una lista de imágenes PIL y devuelve un arreglo de numpy de $n \times 512$. Usa solo lo que trae `ckpt`:
1. `validate_image(img, ckpt['contract'])` para cada imagen, y apilar;
2. normalizar con `ckpt['mean']` y `ckpt['std']`;
3. reconstruir el encoder con `BUILDERS[ckpt['model']]()`, cargar `ckpt['g']` y ponerlo en `eval()`.

In [ ]:
ckpt = torch.load(mlflow.artifacts.download_artifacts(run_id=run_ssl, artifact_path='encoder.pt'), map_location='cpu', weights_only=False)
BUILDERS = {'resnet18_sat': resnet18_sat}

@torch.no_grad()
def embed_raw(ckpt, images):
    # TODO 4
    raise NotImplementedError('Completa TODO 4')

In [ ]:
to_pil = transforms.ToPILImage()
crudas = [to_pil(x) for x in X_test[:500].cpu()]
E = embed_raw(ckpt, crudas)
ref = features(g_ssl, X_test[:500]).cpu().numpy()
cos = (E * ref).sum(1) / (np.linalg.norm(E, axis=1) * np.linalg.norm(ref, axis=1))
print(f'1. Consistencia con features(): coseno mínimo {cos.min():.4f}')
assert E.shape == (len(crudas), 512) and cos.min() > 0.999, 'embed_raw no reproduce h: ¿falta eval() o la normalización?'
for bad in [crudas[0].convert('L'), crudas[0].resize((32, 32))]:
    try:
        embed_raw(ckpt, [bad]); raise AssertionError('Una imagen inválida debió ser rechazada.')
    except ValueError as e:
        print('2. Contrato: imagen rechazada →', e)
np.savez_compressed('embeddings.npz', h=E, y=y_test[:500].cpu().numpy())
print('3. embeddings.npz escrito')

## 9. Preguntas
1. ¿Se cumplió tu predicción de la pérdida inicial? ¿Qué valor da $\ln(2N - 1)$ con $N = 256$?
2. Con **1 %** de las etiquetas, ¿qué encoder ganó? ¿Y con **10 %**? Usa la tabla y explica por qué el orden puede cambiar con la cantidad de etiquetas.
3. ¿Qué transformaciones pusiste en `vista_satelite` y por qué no cambian lo que importa en EuroSAT? ¿Cuál de ellas estaría mal en CIFAR-10?
4. Compara con tu integrador de la sesión 03, que usó **todas** las etiquetas. ¿Cuántas etiquetas te ahorra el mejor encoder de hoy para llegar a una accuracy parecida?
5. En producción, `embed_raw` recibe imágenes de un satélite nuevo con otra resolución de color. ¿Qué parte del artefacto debería detectarlo? ¿Qué harías antes de usar esos embeddings?

_Respuestas:_

**Si quieres ir más allá:** haz *fine-tuning* del encoder SimCLR con el 10 % de las etiquetas (notebook 5, sección 6) y compáralo con la *probe*.